# RAG with OpenAI (Native Implementation)

### Step 1: Setup and Document Preparation

In [1]:
%pip install langchain langchain-community PyPDF2 python-dotenv openai tiktoken pydub pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import necessary libraries
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env file
load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Set default model
MODEL = "gpt-4o-mini"

In [3]:
# Load PDF document and extract its text
import sys
import subprocess

try:
    from PyPDF2 import PdfReader
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "PyPDF2"])
    from PyPDF2 import PdfReader

PDF_PATH = Path("ethics_guidelines_for_trustworthy_ai.pdf")
PDF_TEXT_PATH = Path("ethics_guidelines_for_trustworthy_ai_text.txt")

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF file not found: {PDF_PATH.resolve()}")

pdf_reader = PdfReader(str(PDF_PATH))

pdf_pages = []
for page_number, page in enumerate(pdf_reader.pages, start=1):
    page_text = page.extract_text() or ""
    pdf_pages.append({
        "page": page_number,
        "text": page_text.strip(),
    })

pdf_text = "\n\n".join(
    f"[Page {page['page']}]\n{page['text']}"
    for page in pdf_pages
    if page["text"]
)

PDF_TEXT_PATH.write_text(pdf_text, encoding="utf-8")

print(f"Loaded {len(pdf_pages)} pages from: {PDF_PATH.name}")
print(f"Extracted {len(pdf_text)} characters")
print(f"PDF text saved to: {PDF_TEXT_PATH.resolve()}")

Loaded 41 pages from: ethics_guidelines_for_trustworthy_ai.pdf
Extracted 158394 characters
PDF text saved to: C:\Users\dilia\OneDrive\IronHack\Week2\RAG\RAG_OpenAI\ethics_guidelines_for_trustworthy_ai_text.txt


Chunking PDF file (recursive character + token verification)

In [4]:
import json

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "langchain-text-splitters"])
    from langchain_text_splitters import RecursiveCharacterTextSplitter

try:
    import tiktoken
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tiktoken"])
    import tiktoken

# ── Config ──────────────────────────────────────────────────────────────────
CHUNK_SIZE       = 1000          # characters
CHUNK_OVERLAP    = 200           # characters
SEPARATORS       = ["\n\n", "\n", ". ", " ", ""]
TOKENIZER_MODEL  = "cl100k_base" # GPT-4 / Claude-compatible BPE encoding
TOKEN_WARN_LIMIT = 300           # warn if any chunk exceeds this

CHUNKS_PATH = Path("ethics_guidelines_chunks.json")

# ── Load extracted text ──────────────────────────────────────────────────────
if not PDF_TEXT_PATH.exists():
    raise FileNotFoundError(f"Text file not found: {PDF_TEXT_PATH.resolve()}")

raw_text = PDF_TEXT_PATH.read_text(encoding="utf-8")

# ── Split ────────────────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=SEPARATORS,
    length_function=len,          # character-based splitting
    is_separator_regex=False,
)

raw_chunks = splitter.split_text(raw_text)

# ── Token verification ───────────────────────────────────────────────────────
enc = tiktoken.get_encoding(TOKENIZER_MODEL)

chunks = []
oversized = []

for idx, text in enumerate(raw_chunks, start=1):
    token_count = len(enc.encode(text))
    entry = {
        "chunk_id":    idx,
        "char_count":  len(text),
        "token_count": token_count,
        "text":        text,
    }
    chunks.append(entry)
    if token_count > TOKEN_WARN_LIMIT:
        oversized.append((idx, token_count))

# ── Persist ──────────────────────────────────────────────────────────────────
CHUNKS_PATH.write_text(json.dumps(chunks, ensure_ascii=False, indent=2), encoding="utf-8")

# ── Report ───────────────────────────────────────────────────────────────────
token_counts = [c["token_count"] for c in chunks]

print(f"Total chunks      : {len(chunks)}")
print(f"Avg tokens/chunk  : {sum(token_counts) / len(token_counts):.1f}")
print(f"Min tokens/chunk  : {min(token_counts)}")
print(f"Max tokens/chunk  : {max(token_counts)}")
print(f"Chunks > {TOKEN_WARN_LIMIT} tokens : {len(oversized)}"
      + (f"  ← consider reducing CHUNK_SIZE" if oversized else ""))
print(f"Chunks saved to   : {CHUNKS_PATH.resolve()}")

Total chunks      : 209
Avg tokens/chunk  : 179.8
Min tokens/chunk  : 21
Max tokens/chunk  : 282
Chunks > 300 tokens : 0
Chunks saved to   : C:\Users\dilia\OneDrive\IronHack\Week2\RAG\RAG_OpenAI\ethics_guidelines_chunks.json


Post-filter to review how many of the 209 chunks get dropped.

Note: from the results only 3 chunks were dropped. This might be related to the stray page headers, a lone section number, or a blank-ish page — not a structural issue with the splitting strategy selected. This means that the current setting of (1000 chars / 200 overlap) is working well for this document. 

In [5]:
MIN_TOKENS = 50  # drop chunks too small to be meaningful

chunks = [c for c in chunks if c["token_count"] >= MIN_TOKENS]

print(f"Chunks after min-token filter: {len(chunks)}")

Chunks after min-token filter: 206


### Step 2: Generate Embeddings

In [6]:
import json
import time
import numpy as np
from pathlib import Path
from typing import List

try:
    from openai import OpenAI
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openai"])
    from openai import OpenAI

# ── Config ───────────────────────────────────────────────────────────────────
EMBEDDING_MODEL  = "text-embedding-3-small"  # or "text-embedding-3-large"
BATCH_SIZE       = 100                        # max items per API call
RETRY_ATTEMPTS   = 3
RETRY_DELAY      = 5                          # seconds between retries

CHUNKS_PATH      = Path("ethics_guidelines_chunks.json")
EMBEDDINGS_PATH  = Path("ethics_guidelines_embeddings.json")

# ── Load filtered chunks ─────────────────────────────────────────────────────
chunks = json.loads(CHUNKS_PATH.read_text(encoding="utf-8"))
texts  = [c["text"] for c in chunks]

print(f"Chunks to embed : {len(texts)}")
print(f"Model           : {EMBEDDING_MODEL}")
print(f"Batch size      : {BATCH_SIZE}")
print(f"Estimated calls : {-(-len(texts) // BATCH_SIZE)}")  # ceiling division

# ── Client ───────────────────────────────────────────────────────────────────
client = OpenAI()  # reads OPENAI_API_KEY from environment automatically

# ── Core function ────────────────────────────────────────────────────────────
def get_embeddings_batch(
    texts: List[str],
    model: str = EMBEDDING_MODEL,
    batch_size: int = BATCH_SIZE,
) -> List[List[float]]:
    """Generate embeddings in batches with retry logic."""
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]

        for attempt in range(1, RETRY_ATTEMPTS + 1):
            try:
                response = client.embeddings.create(model=model, input=batch)
                # Sort by index to guarantee order matches input
                batch_embeddings = [
                    item.embedding
                    for item in sorted(response.data, key=lambda x: x.index)
                ]
                all_embeddings.extend(batch_embeddings)
                print(f"  Embedded {min(i + batch_size, len(texts))}/{len(texts)} chunks")
                break  # success — exit retry loop

            except Exception as e:
                print(f"  Attempt {attempt}/{RETRY_ATTEMPTS} failed: {e}")
                if attempt < RETRY_ATTEMPTS:
                    time.sleep(RETRY_DELAY)
                else:
                    raise RuntimeError(f"Embedding failed after {RETRY_ATTEMPTS} attempts") from e

    return all_embeddings

# ── Generate ─────────────────────────────────────────────────────────────────
print("\nGenerating embeddings...")
embeddings = get_embeddings_batch(texts)

# ── Verify dimensions ─────────────────────────────────────────────────────────
embedding_dim = len(embeddings[0])
assert len(embeddings) == len(chunks), "Mismatch between chunks and embeddings count"
print(f"\nEmbedding dimensions : {embedding_dim}")   # 1536 for small, 3072 for large
print(f"Total embeddings     : {len(embeddings)}")

# ── Attach embeddings to chunks and persist ───────────────────────────────────
embedded_chunks = [
    {**chunk, "embedding": embedding}
    for chunk, embedding in zip(chunks, embeddings)
]

EMBEDDINGS_PATH.write_text(
    json.dumps(embedded_chunks, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# ── Summary ───────────────────────────────────────────────────────────────────
matrix = np.array(embeddings)
norms  = np.linalg.norm(matrix, axis=1)

print(f"\nEmbedding norms — mean: {norms.mean():.4f} | min: {norms.min():.4f} | max: {norms.max():.4f}")
print(f"Embeddings saved to  : {EMBEDDINGS_PATH.resolve()}")

Chunks to embed : 209
Model           : text-embedding-3-small
Batch size      : 100
Estimated calls : 3

Generating embeddings...
  Embedded 100/209 chunks
  Embedded 200/209 chunks
  Embedded 209/209 chunks

Embedding dimensions : 1536
Total embeddings     : 209

Embedding norms — mean: 1.0000 | min: 0.9994 | max: 1.0006
Embeddings saved to  : C:\Users\dilia\OneDrive\IronHack\Week2\RAG\RAG_OpenAI\ethics_guidelines_embeddings.json


Note: Sorted by item.index, retry logic and norm sanity check added 

### Step 3: Implement Vector Search

In [7]:
import json
import numpy as np
from pathlib import Path
from typing import List, Dict

# ── Config ───────────────────────────────────────────────────────────────────
EMBEDDINGS_PATH  = Path("ethics_guidelines_embeddings.json")
EMBEDDING_MODEL  = "text-embedding-3-small"
TOP_K            = 5  # number of chunks to retrieve

# ── Load embedded chunks ──────────────────────────────────────────────────────
embedded_chunks = json.loads(EMBEDDINGS_PATH.read_text(encoding="utf-8"))

# Pre-build the embedding matrix once — avoids recomputing on every query
chunk_texts      = [c["text"]      for c in embedded_chunks]
chunk_embeddings = np.array([c["embedding"] for c in embedded_chunks], dtype=np.float32)  # shape: (209, 1536)

print(f"Loaded {len(embedded_chunks)} chunks | Matrix shape: {chunk_embeddings.shape}")

# ── Core functions ────────────────────────────────────────────────────────────
def embed_query(query: str, model: str = EMBEDDING_MODEL) -> np.ndarray:
    """Embed a single query string and return as a numpy vector."""
    response = client.embeddings.create(model=model, input=[query])
    return np.array(response.data[0].embedding, dtype=np.float32)


def cosine_similarity_batch(query_vec: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    """
    Compute cosine similarity between a single query vector and all chunk vectors.
    Since embeddings are unit-normalized, cosine similarity reduces to a dot product.
    Returns a (n_chunks,) array of similarity scores in [-1, 1].
    """
    return matrix @ query_vec  # shape: (209,)


def search(
    query: str,
    top_k: int = TOP_K,
    min_similarity: float = 0.0,  # optional threshold to filter weak matches
) -> List[Dict]:
    """
    Find the top-k most relevant chunks for a given query.

    Returns a list of dicts with keys:
        rank            : 1-based retrieval rank
        chunk_id        : original chunk index
        similarity      : cosine similarity score
        token_count     : chunk token count
        text            : chunk text
    """
    query_vec    = embed_query(query)
    similarities = cosine_similarity_batch(query_vec, chunk_embeddings)

    # Get indices sorted by descending similarity
    ranked_indices = np.argsort(similarities)[::-1]

    results = []
    for rank, idx in enumerate(ranked_indices[:top_k], start=1):
        score = float(similarities[idx])
        if score < min_similarity:
            break
        results.append({
            "rank":        rank,
            "chunk_id":    embedded_chunks[idx]["chunk_id"],
            "similarity":  round(score, 4),
            "token_count": embedded_chunks[idx]["token_count"],
            "text":        chunk_texts[idx],
        })

    return results


def print_results(results: List[Dict], query: str) -> None:
    """Pretty-print search results."""
    print(f"\nQuery: '{query}'")
    print(f"{'─' * 60}")
    for r in results:
        print(f"[Rank {r['rank']}] Chunk {r['chunk_id']:>3} | "
              f"Similarity: {r['similarity']:.4f} | "
              f"Tokens: {r['token_count']}")
        print(f"{r['text'][:300]}{'...' if len(r['text']) > 300 else ''}")
        print()

# ── Test queries ──────────────────────────────────────────────────────────────
test_queries = [
    "What are the principles of trustworthy AI?",
    "How should AI systems handle personal data?",
    "What is human oversight in AI systems?",
]

for query in test_queries:
    results = search(query, top_k=TOP_K)
    print_results(results, query)

Loaded 209 chunks | Matrix shape: (209, 1536)

Query: 'What are the principles of trustworthy AI?'
────────────────────────────────────────────────────────────
[Rank 1] Chunk  24 | Similarity: 0.7403 | Tokens: 201
entire life cycle.  
Trustworthy  AI has three components , which should be met throughout the system's entire life cycle :  
1. it should be lawful , complying with  all applicable laws and regulations ; 
2. it should be ethical , ensuring adherence to  ethical principles and values;  and  
3. it s...

[Rank 2] Chunk 188 | Similarity: 0.7387 | Tokens: 205
confidently and fully reap its benefits when the technology, including the processes and people behind the 
technology, are trustworthy. When drafting these G uidelines, Trustworthy  AI has, therefore, been our foundational 
ambition.  
Trustworthy  AI has thre e components: (1) it should be l awful...

[Rank 3] Chunk  40 | Similarity: 0.7172 | Tokens: 191
[Page 11]
9 
 I. Chapter I: Foundations of Trustworthy  AI  
This Ch

**Notes on vector search:**
* The chunk_embeddings numpy matrix is constructed once when the script loads, not inside the search function. 
* Setting min_similarity=0.3 allows to filter out retrievals where the model is effectively guessing, which matters when a query falls outside the document's domain entirely.
* Dot product instead of full cosine similarity formula. Since your embeddings are unit-normalized (norms ≈ 1.0, confirmed in the previous step), cosine similarity reduces to a simple dot product — matrix @ query_vec. This is both faster and numerically identical to the full formula, avoiding redundant norm divisions.


### Step 4: Build RAG Query Function

Option1: no UI

In [8]:
from openai import OpenAI
from typing import List, Dict

# ── Config ────────────────────────────────────────────────────────────────────
CHAT_MODEL       = "gpt-4o-mini"   # or "gpt-4o" for higher quality
MAX_TOKENS       = 1000
TEMPERATURE      = 0.2             # low = more factual, less creative
TOP_K            = 5

SYSTEM_PROMPT = """You are an expert assistant specialized in AI ethics and trustworthy AI guidelines.
Answer questions based strictly on the provided context chunks retrieved from the document.
If the answer cannot be found in the context, say so clearly — do not hallucinate or infer beyond what is given.
Always be concise, precise, and cite which part of the context supports your answer."""

# ── Context formatter ─────────────────────────────────────────────────────────
def format_context(chunks: List[Dict]) -> str:
    """Format retrieved chunks into a structured context block for the LLM."""
    sections = []
    for c in chunks:
        sections.append(
            f"[Chunk {c['chunk_id']} | Similarity: {c['similarity']} | Tokens: {c['token_count']}]\n"
            f"{c['text']}"
        )
    return "\n\n---\n\n".join(sections)


# ── Core RAG function ─────────────────────────────────────────────────────────
def rag_query(
    query: str,
    top_k: int = TOP_K,
    min_similarity: float = 0.3,
    verbose: bool = True,
) -> Dict:
    """
    Full RAG pipeline:
      1. Embed query
      2. Retrieve top-k similar chunks
      3. Format context
      4. Generate answer via Chat Completions

    Returns a dict with keys: query, answer, chunks_used, model
    """
    # Step 1 — Retrieve
    chunks = search(query, top_k=top_k, min_similarity=min_similarity)

    if not chunks:
        return {
            "query":       query,
            "answer":      "No sufficiently relevant context found in the document.",
            "chunks_used": 0,
            "model":       CHAT_MODEL,
        }

    if verbose:
        print(f"\n📥 Query      : {query}")
        print(f"📦 Chunks used: {len(chunks)} "
              f"(similarity range: {chunks[-1]['similarity']} – {chunks[0]['similarity']})")

    # Step 2 — Format context
    context = format_context(chunks)

    # Step 3 — Generate
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": (
                f"Context:\n\n{context}\n\n"
                f"Question: {query}"
            )},
        ],
    )

    answer = response.choices[0].message.content

    if verbose:
        print(f"\n🤖 Answer:\n{answer}")
        print(f"\n📊 Usage — prompt: {response.usage.prompt_tokens} tokens | "
              f"completion: {response.usage.completion_tokens} tokens | "
              f"total: {response.usage.total_tokens} tokens")

    return {
        "query":       query,
        "answer":      answer,
        "chunks_used": len(chunks),
        "model":       CHAT_MODEL,
        "usage":       response.usage,
    }


# ── Run ───────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    test_queries = [
        "What are the principles of trustworthy AI?",
        "How should AI systems handle personal data?",
        "What is human oversight in AI systems?",
    ]

    for query in test_queries:
        result = rag_query(query)
        print("\n" + "═" * 60)


📥 Query      : What are the principles of trustworthy AI?
📦 Chunks used: 5 (similarity range: 0.7031 – 0.7403)

🤖 Answer:
The principles of trustworthy AI are: 

1. It should be lawful, ensuring compliance with all applicable laws and regulations.
2. It should be ethical, ensuring adherence to ethical principles and values.
3. It should be robust, both from a technical and social perspective, to prevent unintentional harm.

This is supported by the context in both Chunk 24 and Chunk 188, which outline these three components as necessary for achieving trustworthy AI.

📊 Usage — prompt: 1193 tokens | completion: 89 tokens | total: 1282 tokens

════════════════════════════════════════════════════════════

📥 Query      : How should AI systems handle personal data?
📦 Chunks used: 5 (similarity range: 0.6066 – 0.6855)

🤖 Answer:
AI systems should handle personal data by ensuring privacy and data protection throughout the system's entire lifecycle. This includes:

1. Guaranteeing that data c

Notes:
* Low temperature (0.2). You want factual, document-grounded answers — not creative ones. Keeping temperature low reduces hallucination risk significantly in RAG settings.
* min_similarity=0.3 as the floor. Chunks below this score are dropped before being sent to the LLM. Sending weak context is worse than sending less context — it introduces noise that can mislead the model.
* Context formatted with metadata. Each chunk is labeled with its ID and similarity score before being passed to the LLM. This helps the model weight higher-similarity chunks more heavily and gives you traceability in the answer.
* Gradio sliders for Top-K and similarity. These are the two most impactful retrieval parameters, and exposing them in the UI lets you tune retrieval quality interactively without touching code.

Option2: Interactive chat UI with Gradio - cannot test (my notebook at the moment is too slow)

In [10]:
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "gradio"])

import gradio as gr
from typing import List, Tuple

c:\Users\dilia\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import subprocess
import sys
import gradio as gr
from typing import List, Dict, Tuple

# ── Install gradio if missing ─────────────────────────────────────────────────
subprocess.check_call([sys.executable, "-m", "pip", "install", "gradio"])

# ── Config ────────────────────────────────────────────────────────────────────
CHAT_MODEL  = "gpt-4o-mini"
MAX_TOKENS  = 1000
TEMPERATURE = 0.2
TOP_K       = 5

SYSTEM_PROMPT = """You are an expert assistant specialized in AI ethics and trustworthy AI guidelines.
Answer questions based strictly on the provided context chunks retrieved from the document.
If the answer cannot be found in the context, say so clearly — do not hallucinate or infer beyond what is given.
Always be concise, precise, and cite which part of the context supports your answer."""

# ── Context formatter ─────────────────────────────────────────────────────────
def format_context(chunks: List[Dict]) -> str:
    """Format retrieved chunks into a structured context block for the LLM."""
    sections = []
    for c in chunks:
        sections.append(
            f"[Chunk {c['chunk_id']} | Similarity: {c['similarity']} | Tokens: {c['token_count']}]\n"
            f"{c['text']}"
        )
    return "\n\n---\n\n".join(sections)


# ── Core RAG function ─────────────────────────────────────────────────────────
def rag_query(
    query: str,
    top_k: int = TOP_K,
    min_similarity: float = 0.3,
    verbose: bool = True,
) -> Dict:
    """
    Full RAG pipeline:
      1. Embed query
      2. Retrieve top-k similar chunks
      3. Format context
      4. Generate answer via Chat Completions
    """
    chunks = search(query, top_k=top_k, min_similarity=min_similarity)

    if not chunks:
        return {
            "query":       query,
            "answer":      "No sufficiently relevant context found in the document.",
            "chunks_used": 0,
            "model":       CHAT_MODEL,
            "usage":       None,
        }

    if verbose:
        print(f"\n📥 Query      : {query}")
        print(f"📦 Chunks used: {len(chunks)} "
              f"(similarity range: {chunks[-1]['similarity']} – {chunks[0]['similarity']})")

    context  = format_context(chunks)
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": (
                f"Context:\n\n{context}\n\n"
                f"Question: {query}"
            )},
        ],
    )

    answer = response.choices[0].message.content

    if verbose:
        print(f"\n🤖 Answer:\n{answer}")
        print(f"\n📊 Usage — prompt: {response.usage.prompt_tokens} tokens | "
              f"completion: {response.usage.completion_tokens} tokens | "
              f"total: {response.usage.total_tokens} tokens")

    return {
        "query":       query,
        "answer":      answer,
        "chunks_used": len(chunks),
        "model":       CHAT_MODEL,
        "usage":       response.usage,
    }


# ── Gradio chat wrapper ───────────────────────────────────────────────────────
def chat_with_rag(
    message: str,
    history: List[Dict],
    top_k: int,
    min_similarity: float,
) -> Tuple[str, List[Dict]]:
    """Gradio-compatible RAG wrapper with dict-format chat history."""
    result = rag_query(
        message,
        top_k=int(top_k),
        min_similarity=min_similarity,
        verbose=False,
    )

    usage_line = (
        f"Tokens used: {result['usage'].total_tokens}"
        if result["usage"] else "No chunks retrieved"
    )

    full_answer = (
        result["answer"]
        + f"\n\n---\n📦 *Retrieved {result['chunks_used']} chunks · "
        f"Model: {result['model']} · {usage_line}*"
    )

    # ✅ Append as dicts — required by Gradio messages format
    history.append({"role": "user",      "content": message})
    history.append({"role": "assistant", "content": full_answer})

    return "", history


# ── UI layout ─────────────────────────────────────────────────────────────────
with gr.Blocks(title="RAG — EU AI Ethics Guidelines", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 📘 RAG Assistant — EU AI Ethics Guidelines
    Ask any question about the *Ethics Guidelines for Trustworthy AI* document.
    Answers are grounded strictly in the document's content.
    """)

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                height=500,
                label="Chat",
                type="messages",    # ✅ enables dict-format message handling
            )
            msg = gr.Textbox(
                placeholder="Ask a question about the document...",
                label="Your question",
                lines=2,
            )
            with gr.Row():
                submit_btn = gr.Button("Send", variant="primary")
                clear_btn  = gr.Button("Clear")

        with gr.Column(scale=1):
            gr.Markdown("### ⚙️ Retrieval settings")
            top_k_slider = gr.Slider(
                minimum=1, maximum=10, value=5, step=1,
                label="Top-K chunks",
                info="Number of chunks retrieved per query",
            )
            sim_slider = gr.Slider(
                minimum=0.0, maximum=0.9, value=0.3, step=0.05,
                label="Min similarity threshold",
                info="Filter out chunks below this score",
            )
            gr.Markdown("### 💡 Example questions")
            gr.Examples(
                examples=[
                    "What are the principles of trustworthy AI?",
                    "How should AI systems handle personal data?",
                    "What is human oversight in AI systems?",
                    "What are the risks of biased AI?",
                    "How is transparency defined in the guidelines?",
                ],
                inputs=msg,
            )

    history_state = gr.State([])

    # ── Event bindings ────────────────────────────────────────────────────────
    submit_btn.click(
        fn=chat_with_rag,
        inputs=[msg, history_state, top_k_slider, sim_slider],
        outputs=[msg, chatbot],
    )
    msg.submit(
        fn=chat_with_rag,
        inputs=[msg, history_state, top_k_slider, sim_slider],
        outputs=[msg, chatbot],
    )
    clear_btn.click(
        lambda: ([], []),
        outputs=[chatbot, history_state],
    )

demo.launch(share=False)